# Week 48, Day 3: Measuring the prior: inference from psychophysics

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w048/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Two-alternative choices from a Bayesian observer

Complete choose(): draw measurements for test and reference, shift each by a sigma^2, and return whether the test's estimate is larger. Simulate 80 trials at each of nine log-speed differences du from -0.6 to 0.6 for the high- and low-contrast test.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import minimize
r = np.random.default_rng(2006)
sig = {"high": 0.15, "low": 0.35}; a_true = -1.2
u_ref = np.log(4.0)
du = np.linspace(-0.6, 0.6, 9); n_tr = 80

def choose(u_test, c_test, n):
    m_t = u_test + sig[c_test] * r.normal(size=n)
    m_r = u_ref + sig["high"] * r.normal(size=n)
    ...

k_high = np.array([choose(u_ref + d, "high", n_tr).sum() for d in du])
k_low = np.array([choose(u_ref + d, "low", n_tr).sum() for d in du])
p_high, p_low = k_high / n_tr, k_low / n_tr
plt.plot(du, p_high, "o", label="high-contrast test"); plt.plot(du, p_low, "o", label="low-contrast test")
plt.axhline(0.5, color="k", ls=":"); plt.xlabel("log speed of test minus reference"); plt.ylabel("P(test faster)"); plt.legend(); plt.show()
print("P(test faster) at equal speeds: high", p_high[4], " low", p_low[4])

In [ ]:
import numpy as np
def test_shape():
    "the high-contrast test is near 50% at the reference speed and the low-contrast test is chosen less often across the range"
    mid = len(du) // 2
    assert abs(p_high[mid] - 0.5) < 0.12 and p_low[mid] < 0.5 and p_low[mid + 1] < p_high[mid + 1]

test_shape()
print("All checks passed")

### Exercise: PSE and width by maximum likelihood

Write nll(params, k): the negative binomial log-likelihood of the counts k at the differences du, for P = Phi((du - mu) / s) with s = exp(log_s) to keep it positive. Fit both conditions with Nelder-Mead from (0, log 0.3).

In [ ]:
def nll(params, k):
    mu, log_s = params
    p = np.clip(norm.cdf((du - mu) / np.exp(log_s)), 1e-6, 1 - 1e-6)
    return ...

fits = {}
for name, k in [("high", k_high), ("low", k_low)]:
    res = minimize(nll, [0.0, np.log(0.3)], args=(k,), method="Nelder-Mead")
    fits[name] = (res.x[0], np.exp(res.x[1]))
    print(f"{name}-contrast test: PSE {fits[name][0]:+.3f}, width {fits[name][1]:.3f}")

In [ ]:
import numpy as np
def test_widths():
    "fitted widths are within 0.06 of their true values sqrt(sig_t^2 + sig_r^2)"
    assert abs(fits["high"][1] - np.sqrt(2) * 0.15) < 0.06 and abs(fits["low"][1] - np.sqrt(0.35 ** 2 + 0.15 ** 2)) < 0.08
def test_pse():
    "the low-contrast PSE is shifted positive by about -a(sig_l^2 - sig_h^2) = 0.12"
    assert 0.05 < fits["low"][0] < 0.22 and abs(fits["high"][0]) < 0.06

test_widths()
test_pse()
print("All checks passed")

### Exercise: Noise and prior slope from two fits

From the high-contrast width, sig_h = s_high / sqrt 2. From the low-contrast width, sig_l = sqrt(s_low^2 - sig_h^2). Then the prior slope is a = -PSE_low / (sig_l^2 - sig_h^2).

In [ ]:
sig_h = ...
sig_l = ...
a_hat = ...
print(f"recovered noise: high contrast {sig_h:.3f} (true 0.15), low contrast {sig_l:.3f} (true 0.35); prior slope {a_hat:.2f} (true -1.2)")

In [ ]:
import numpy as np
def test_sigmas():
    "recovered noise levels are within 0.05 of 0.15 and 0.35"
    assert abs(sig_h - 0.15) < 0.05 and abs(sig_l - 0.35) < 0.07
def test_slope():
    "the recovered prior slope is negative and within 0.6 of -1.2"
    assert a_hat < 0 and abs(a_hat + 1.2) < 0.6

test_sigmas()
test_slope()
print("All checks passed")